# SMC 策略可视化 — 决策流程解析

逐步展示回测引擎每个决策阶段，配合实际 SNDK 数据可视化。

| 步骤 | 函数 | 作用 |
|------|------|------|
| 1 | `find_swings` | 检测摆动高低点序列 |
| 2 | `detect_bos_choch` | 识别 BOS / CHoCH 市场结构突破 |
| 3 | `determine_trend` | 基于连续 BOS 判断趋势方向 |
| 4 | `detect_fvg` | 找出 FVG 公允价值缺口 |
| 5 | `fvg_entry_depth` | 检查价格填入 FVG 的深度 |
| 6 | `check_ltf_confirmation` | LTF CHoCH + BOS 入场信号 |
| 7 | 交易管理 | SL / TP 计算、RR 过滤、开仓 |

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

import builtins
_nb = builtins.__dict__.get('__vsc_ipynb_file__')

def _find_root(nb_path=None):
    candidates = list(pathlib.Path(nb_path).parents) if nb_path else []
    candidates += [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents)
    for p in candidates:
        if (p / 'pyproject.toml').exists():
            return p
    return None

_root = _find_root(_nb)
if _root is None:
    raise RuntimeError(f'Cannot find project root. CWD={pathlib.Path.cwd()}')
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root : {_root}')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print(f'matplotlib   : {matplotlib.__version__}  backend={matplotlib.get_backend()}')

BG, BG2  = '#0b1120', '#131f30'
FG, GRID = '#cdd6f4', '#1e2d42'
GREEN, RED, GOLD = '#26a69a', '#ef5350', '#f9a825'
BLUE, PURPLE, CYAN = '#5c9cf5', '#ce93d8', '#80cbc4'

plt.rcParams.update({
    'figure.facecolor': BG,  'axes.facecolor': BG2,
    'axes.edgecolor':   GRID, 'text.color':     FG,
    'axes.labelcolor':  FG,   'xtick.color':    FG,
    'ytick.color':      FG,   'grid.color':     GRID,
    'grid.linewidth':   0.5,  'grid.alpha':     0.5,
    'grid.linestyle':   '--', 'font.size':      8,
})

def draw_candles(ax, df, x0=0):
    df = df.reset_index(drop=True)
    for i, r in df.iterrows():
        x  = i + x0
        up = r['close'] >= r['open']
        c  = GREEN if up else RED
        lo = min(r['open'], r['close'])
        hi = max(r['open'], r['close'])
        ax.add_patch(mpatches.Rectangle(
            (x - 0.38, lo), 0.76, max(hi - lo, 1e-8),
            fc=c, ec=c, alpha=0.85, zorder=2))
        ax.plot([x, x], [r['low'], lo],  c, lw=0.7, zorder=1)
        ax.plot([x, x], [hi, r['high']], c, lw=0.7, zorder=1)
    pad = (df['high'].max() - df['low'].min()) * 0.025
    ax.set_xlim(x0 - 1, x0 + len(df))
    ax.set_ylim(df['low'].min() - pad, df['high'].max() + pad)

def style_ax(ax, title=''):
    ax.set_facecolor(BG2)
    ax.tick_params(colors=FG, labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.yaxis.label.set_color(FG)
    ax.xaxis.label.set_color(FG)
    ax.grid(color=GRID, lw=0.5, ls='--', alpha=0.5)
    if title:
        ax.set_title(title, color=FG, fontsize=9, fontweight='bold', pad=5)

def time_ticks(ax, df, step=10, x0=0, fmt='%m/%d %H:%M'):
    idx = list(range(0, len(df), step))
    ax.set_xticks([i + x0 for i in idx])
    ax.set_xticklabels(
        [pd.to_datetime(df.iloc[i]['time_key']).strftime(fmt) for i in idx],
        rotation=30, ha='right', fontsize=6)

print('Setup complete')

## 数据加载

In [ ]:
import sys, pathlib, importlib

_nb = globals().get('__vsc_ipynb_file__')
_root = None

if _nb:
    for _p in pathlib.Path(_nb).parents:
        if (_p / 'pyproject.toml').exists():
            _root = _p
            break

if _root is None:
    for _p in [pathlib.Path.cwd()] + list(pathlib.Path.cwd().parents):
        if (_p / 'pyproject.toml').exists():
            _root = _p
            break

if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    print('Project root:', _root)
else:
    print('ERROR: pyproject.toml not found, CWD:', pathlib.Path.cwd())

# Purge all project modules so each kernel run starts clean.
# Prevents stale sys.modules from a previous failed import causing ImportError.
_PROJECT_PKGS = ('strategy', 'backtest', 'feeds', 'core', 'analysis')
_stale = [k for k in list(sys.modules) if k.split('.')[0] in _PROJECT_PKGS]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print('Cleared', len(_stale), 'stale module(s)')

import strategy.smc
import backtest.engine
import feeds.fetcher

from feeds.fetcher import fetch_klines
from strategy.smc import (
    find_swings, detect_bos_choch, detect_fvg,
    determine_trend, fvg_entry_depth,
    check_ltf_confirmation, is_displacement_candle,
)
from backtest.engine import BacktestParams, run_backtest

# Global displacement parameters
_DISP_MULT  = 1.5
_BODY_RATIO = 0.5

CODE, START, END = 'US.SNDK', '2025-02-13', '2025-12-31'
htf_raw = fetch_klines(CODE, '60m', START, END)
ltf_raw = fetch_klines(CODE, '15m', START, END)

print('HTF 60m :', len(htf_raw), 'bars ', htf_raw['time_key'].iloc[0], '->', htf_raw['time_key'].iloc[-1])
print('LTF 15m :', len(ltf_raw), 'bars ', ltf_raw['time_key'].iloc[0], '->', ltf_raw['time_key'].iloc[-1])

params = BacktestParams(trend_tf='60m', entry_tf='15m',
                        swing_lookback=2, bos_count=1,
                        fvg_min_width_pct=0.002, fvg_entry_depth_pct=0.20,
                        sl_buffer_pct=0.001, max_sl_pct=0.010, min_rr=1.5)
result = run_backtest(htf_raw, ltf_raw, params)

s = result.summary_dict()
print('Backtest:', s['n_trades'], 'trades  WR=' + str(round(s['win_rate']*100)) + '%  TotalR=' + str(round(s['total_r'],2)) + '  PF=' + str(round(s['profit_factor'],2)))
for i, t in enumerate(result.trades):
    print(f'  #{i+1} {t.direction}  entry={t.entry_time}  SL={t.sl:.2f}  TP={t.tp:.2f}  RR={t.planned_rr:.2f}  R={t.r_multiple:+.2f}  [{t.result}]')

## 决策流程总览

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10), facecolor=BG)
ax.set_facecolor(BG); ax.axis('off')
ax.set_xlim(0, 10); ax.set_ylim(-0.5, 14.5)

_steps = [
    ('1  HTF Swing Points',    'find_swings(htf, lookback)',                          BLUE),
    ('2  BOS / CHoCH',         'detect_bos_choch(htf, lookback)',                    PURPLE),
    ('3  Trend Direction',     'determine_trend(bos_signals, min_consecutive)',       GOLD),
    ('4  FVG + Entry Depth',   'detect_fvg(htf)  ->  price in zone, depth >= thr',  CYAN),
    ('5  LTF CHoCH + BOS',     'check_ltf_confirmation(ltf_bos, trend)',             GREEN),
    ('6  SL / TP / RR Filter', 'SL = swing +/- buffer   TP = opposing swing',        GOLD),
    ('7  Open Trade',          'Trade(direction, entry_price, sl, tp)',              GREEN),
]
ys = [13, 11, 9, 7, 5, 3, 1]
for y, (title, code, color) in zip(ys, _steps):
    ax.text(5, y + 0.1, title, color=color, fontsize=10, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.45', fc=BG2, ec=color, lw=1.8))
    ax.text(5, y - 0.6, code, color=FG, fontsize=7.5, ha='center', va='top',
            alpha=0.75, fontfamily='monospace')
for ya, yb in zip(ys[:-1], ys[1:]):
    ax.annotate('', xy=(5, yb + 0.48), xytext=(5, ya - 0.78),
                arrowprops=dict(arrowstyle='->', color=GRID, lw=1.5))

ax.set_title('SMC Strategy -- Decision Filter Chain', color=FG, fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## Step 1 — HTF 摆动高低点 (`find_swings`)

第 `i` 根K线，若其高点是前后 `lookback` 根K线中最高的 → 摆动高点（▲）；低点同理（▼）。
然后强制交替排列（高→低→高→低…）。

`lookback` 越大 → 识别越平滑，但信号越稀疏。

In [ ]:
htf = htf_raw.tail(80).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), facecolor=BG)
for ax, lb in zip(axes, [2, 4]):
    swings = find_swings(htf, lookback=lb)
    draw_candles(ax, htf)
    style_ax(ax, f'lookback = {lb}')
    time_ticks(ax, htf, step=12)

    for sw in swings:
        if sw['kind'] == 'high':
            ax.plot(sw['idx'], sw['price'], '^', color=GREEN, ms=7, zorder=5)
            ax.text(sw['idx'], sw['price'] * 1.003, f"{sw['price']:.1f}",
                    color=GREEN, fontsize=5, ha='center', va='bottom')
        else:
            ax.plot(sw['idx'], sw['price'], 'v', color=RED, ms=7, zorder=5)
            ax.text(sw['idx'], sw['price'] * 0.997, f"{sw['price']:.1f}",
                    color=RED, fontsize=5, ha='center', va='top')

    xs = [s['idx'] for s in swings]
    ys = [s['price'] for s in swings]
    ax.plot(xs, ys, color=GRID, lw=0.7, ls=':', zorder=3)

    nh = sum(1 for s in swings if s['kind'] == 'high')
    nl = sum(1 for s in swings if s['kind'] == 'low')
    ax.set_xlabel(f'^ Swing High x{nh}    v Swing Low x{nl}', fontsize=7)
    ax.set_ylabel('Price', fontsize=7)

plt.suptitle('Step 1: HTF Swing Point Detection', color=FG, fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 2 — BOS / CHoCH (`detect_bos_choch`)

| 信号 | 条件 | 含义 |
|------|------|------|
| **BOS ↑** | 顺势突破前摆动高点 | 上涨结构延续 |
| **BOS ↓** | 顺势跌破前摆动低点 | 下跌结构延续 |
| **CHoCH ↑** | 下跌途中突破前高 | 结构反转 → 转牛 |
| **CHoCH ↓** | 上涨途中跌破前低 | 结构反转 → 转熊 |

实线 = BOS，虚线 = CHoCH。

In [ ]:
swings  = find_swings(htf, lookback=2)
signals = detect_bos_choch(htf, lookback=2)

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
draw_candles(ax, htf)
style_ax(ax, 'Step 2: BOS / CHoCH -- Market Structure Breaks')
time_ticks(ax, htf, step=10)

for sw in swings:
    m = '^' if sw['kind'] == 'high' else 'v'
    c = GREEN if sw['kind'] == 'high' else RED
    ax.plot(sw['idx'], sw['price'], m, color=c, ms=6, zorder=5)

for sig in signals:
    bull  = sig['direction'] == 'bull'
    color = GREEN if bull else RED
    ls    = '-' if sig['type'] == 'BOS' else '--'
    ax.hlines(sig['price'], sig['from_idx'], sig['idx'],
              colors=color, linestyles=ls, linewidths=1.1, alpha=0.85, zorder=4)
    mid_x = (sig['from_idx'] + sig['idx']) / 2
    off   = sig['price'] * 0.0015 * (1 if bull else -1)
    ax.text(mid_x, sig['price'] + off, sig['type'],
            color=color, fontsize=6, ha='center',
            va='bottom' if bull else 'top', fontweight='bold')

items = [
    mpatches.Patch(fc=GREEN, label='BOS up (bullish break of structure)'),
    mpatches.Patch(fc=RED,   label='BOS down (bearish break of structure)'),
    mpatches.Patch(fc=GREEN, alpha=0.4, label='CHoCH up (reversal -> bull)'),
    mpatches.Patch(fc=RED,   alpha=0.4, label='CHoCH down (reversal -> bear)'),
]
leg = ax.legend(handles=items, fontsize=6.5, framealpha=0.3, labelcolor=FG, loc='upper left')
leg.get_frame().set_facecolor(BG2)
ax.set_ylabel('Price', fontsize=7)
plt.tight_layout()
plt.show()


## Step 3 — 趋势判断 (`determine_trend`)

遍历 BOS/CHoCH 序列：
- **CHoCH** → 重置方向，连续计数归 1
- 同向 **BOS** → 计数 +1
- 计数 ≥ `min_consecutive` 才输出有效趋势

背景色：绿 = 当前趋势判断为牛，红 = 熊，灰 = 无确定趋势。

In [ ]:
print(f'{"idx":>5}  {"type":>6}  {"dir":>5}  {"price":>8}   min=1      min=2')
print('-' * 58)
t1 = t2 = None; c1 = c2 = 0
for sig in sorted(signals, key=lambda s: s['idx']):
    if sig['type'] == 'CHoCH':
        t1 = t2 = sig['direction']; c1 = c2 = 1
    elif sig['type'] == 'BOS':
        if sig['direction'] == t1: c1 += 1
        if sig['direction'] == t2: c2 += 1
    o1 = (t1 or '-') if (t1 and c1 >= 1) else '-'
    o2 = (t2 or '-') if (t2 and c2 >= 2) else '-'
    print(f"{sig['idx']:>5}  {sig['type']:>6}  {sig['direction']:>5}  "
          f"{sig['price']:>8.2f}   {o1:>9}  {o2:>9}")

fig, ax = plt.subplots(figsize=(16, 4.5), facecolor=BG)
draw_candles(ax, htf)
style_ax(ax, 'Step 3: Trend State (min_consecutive=1, background = confirmed trend)')
time_ticks(ax, htf, step=10)

# Mark no-confirmed-trend zone (before first CHoCH) with gray
_sigs_sorted = sorted(signals, key=lambda s: s['idx'])
_first_choch = next((s['idx'] for s in _sigs_sorted if s['type'] == 'CHoCH'), None)
if _first_choch:
    ax.axvspan(0, _first_choch, facecolor='#888888', alpha=0.10)
    ax.text(_first_choch / 2, ax.get_ylim()[1] * 0.998,
            'no confirmed trend', color='#aaaaaa', fontsize=5.5, ha='center', va='top')

cur = None; prev_x = 0
for sig in _sigs_sorted:
    if cur is not None:
        c = GREEN if cur == 'bull' else RED
        ax.axvspan(prev_x, sig['idx'], facecolor=c, alpha=0.15)
    if sig['type'] == 'CHoCH':
        cur = sig['direction']
        color = GREEN if cur == 'bull' else RED
        ax.axvline(sig['idx'], color=color, lw=1.3, ls='--', alpha=0.9)
        ax.text(sig['idx'] + 0.4, ax.get_ylim()[1] * 0.999,
                'CHoCH\n' + cur, color=color, fontsize=6, va='top')
    prev_x = sig['idx']
if cur:
    ax.axvspan(prev_x, len(htf), facecolor=GREEN if cur == 'bull' else RED, alpha=0.15)

items = [mpatches.Patch(fc='#888888', alpha=0.3, label='No confirmed trend (BOS only)'),
         mpatches.Patch(fc=GREEN, alpha=0.4, label='Bull zone (after CHoCH bull)'),
         mpatches.Patch(fc=RED,   alpha=0.4, label='Bear zone (after CHoCH bear)')]
leg = ax.legend(handles=items, fontsize=7, loc='upper left', framealpha=0.3, labelcolor=FG)
leg.get_frame().set_facecolor(BG2)
ax.set_ylabel('Price', fontsize=7)
plt.tight_layout()
plt.show()


## Step 4 — FVG 公允价值缺口 (`detect_fvg`)

**三根K线模式**：
- **牛FVG**：`high[i-2] < low[i]`，中间K线向上跳空留下缺口 → 绿色区域
- **熊FVG**：`low[i-2] > high[i]`，中间K线向下跳空留下缺口 → 红色区域

价格**回拉进入 FVG** 时寻找入场信号（SMC 核心逻辑）。
浅色 = 已被后续K线填满（失效），深色 = 仍有效。

In [ ]:
fvgs = detect_fvg(htf, min_gap_pct=0.002)

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
draw_candles(ax, htf)
n_active = sum(1 for f in fvgs if not f['filled'])
style_ax(ax, f'Step 4: Fair Value Gaps  ({len(fvgs)} total, {n_active} active, {len(fvgs)-n_active} filled)')
time_ticks(ax, htf, step=10)

n_disp = 0
for fvg in fvgs:
    is_disp = is_displacement_candle(htf, fvg['idx'], _DISP_MULT, _BODY_RATIO)
    color   = GREEN if fvg['direction'] == 'bull' else RED
    h       = fvg['top'] - fvg['bottom']
    x0_fvg  = fvg['idx'] - 2
    x_end   = len(htf) - x0_fvg

    if fvg['filled']:
        # ghost: short fixed-width zone at formation point only — gray dashed outline
        ghost_w = min(8, x_end)
        ax.add_patch(mpatches.Rectangle(
            (x0_fvg, fvg['bottom']), ghost_w, h,
            fc='#3a3a4a', ec='#7a7a8a', alpha=0.50, lw=0.6,
            linestyle='--', zorder=1))
    elif is_disp:
        # displacement: solid fill + thick same-color border + white label
        ax.add_patch(mpatches.Rectangle(
            (x0_fvg, fvg['bottom']), x_end, h,
            fc=color, ec=color, alpha=0.40, lw=2.0, zorder=2))
        ax.hlines([fvg['bottom'], fvg['top']], x0_fvg, len(htf),
                  colors=color, lw=1.2, ls='--', alpha=0.85)
        mid = (fvg['top'] + fvg['bottom']) / 2
        ax.text(fvg['idx'] + 0.5, mid,
                fvg['direction'][0].upper() + ' FVG',
                color='white', fontsize=5.5, va='center', fontweight='bold',
                bbox=dict(fc=color, ec='none', alpha=0.70, pad=1))
        n_disp += 1
    else:
        # no displacement: gold border + light color fill — clearly distinct from candles
        ax.add_patch(mpatches.Rectangle(
            (x0_fvg, fvg['bottom']), x_end, h,
            fc=color, ec=GOLD, alpha=0.20, lw=1.8, zorder=1))
        ax.hlines([fvg['bottom'], fvg['top']], x0_fvg, len(htf),
                  colors=GOLD, lw=0.9, ls=':', alpha=0.70)
        mid = (fvg['top'] + fvg['bottom']) / 2
        ax.text(fvg['idx'] + 0.5, mid,
                fvg['direction'][0].upper() + ' no-disp',
                color=GOLD, fontsize=5.0, va='center',
                bbox=dict(fc=BG2, ec='none', alpha=0.60, pad=1))

n_bull = sum(1 for f in fvgs if f['direction'] == 'bull' and not f['filled'])
n_bear = sum(1 for f in fvgs if f['direction'] == 'bear' and not f['filled'])
ax.set_xlabel(f'Active — Bull={n_bull}  Bear={n_bear}  Displacement={n_disp}  (filled=ghost)', fontsize=7)

items = [
    mpatches.Patch(fc=GREEN, ec=GREEN, alpha=0.6, lw=2.0, label='Bull FVG — displacement'),
    mpatches.Patch(fc=GREEN, ec=GOLD,  alpha=0.4, lw=1.8, label='Bull FVG — no displacement'),
    mpatches.Patch(fc=RED,   ec=RED,   alpha=0.6, lw=2.0, label='Bear FVG — displacement'),
    mpatches.Patch(fc=RED,   ec=GOLD,  alpha=0.4, lw=1.8, label='Bear FVG — no displacement'),
    mpatches.Patch(fc='#3a3a4a', ec='#7a7a8a', alpha=0.6, lw=0.6, label='Filled (ghost — 8-bar window)'),
]
leg = ax.legend(handles=items, fontsize=6.5, framealpha=0.3, labelcolor=FG, loc='upper left')
leg.get_frame().set_facecolor(BG2)
ax.set_ylabel('Price', fontsize=7)
plt.tight_layout()
plt.show()


## Step 5 — FVG Entry Depth (`fvg_entry_depth`)

FVG 进入以**影线**为准：

- **牛FVG**（价格从上方回落）：用 `bar_low`（下影线）判断是否触及 FVG，`depth = (top − low) / size`
- **熊FVG**（价格从下方反弹）：用 `bar_high`（上影线）判断是否触及 FVG，`depth = (high − bottom) / size`

`depth=0` 刚碰到边缘，`depth=1` 影线完全穿透（此时 clamp 为 1.0）。  
参数 `fvg_entry_depth_pct=0.20` 要求影线至少穿入 20%。  
收盘价不需要进入 FVG —— 影线触及即触发后续确认流程。

In [ ]:
_DISP_MULT  = 1.5   # range multiplier — matches BacktestParams.displacement_atr_mult
_BODY_RATIO = 0.5   # body/range minimum — matches BacktestParams.displacement_body_ratio
_MIN_DEPTH = 0.05  # wick must enter at least this far into the FVG
_MAX_DEPTH = 0.99  # exclude full-penetration (depth=100%) from demo
_WARMUP    = 20    # skip first N bars — dataset edge may have incomplete data

# Search htf_raw for a displacement FVG with a qualifying wick entry.
htf_demo = htf_raw.reset_index(drop=True)
all_fvgs  = detect_fvg(htf_demo, min_gap_pct=0.002)

demo = None
for f in all_fvgs:
    if f['idx'] < _WARMUP:          # skip dataset warmup region
        continue
    if not is_displacement_candle(htf_demo, f['idx'], _DISP_MULT, _BODY_RATIO):
        continue
    _idx, _bot, _top = f['idx'], f['bottom'], f['top']
    _bull = f['direction'] == 'bull'
    for j in range(_idx + 1, min(len(htf_demo), _idx + 30)):
        r   = htf_demo.iloc[j]
        wp  = r['low'] if _bull else r['high']
        ov  = (_bull and wp <= _top and r['high'] >= _bot) or \
              (not _bull and wp >= _bot and r['low'] <= _top)
        if ov and _MIN_DEPTH <= fvg_entry_depth(f, wp) <= _MAX_DEPTH:
            demo = f
            break
    if demo is not None:
        break

# Final fallback: first unfilled FVG in the 80-bar window
if demo is None:
    demo = next((f for f in detect_fvg(htf, min_gap_pct=0.002) if not f['filled']), None)
    htf_demo = htf

if demo is None:
    print('No FVG available for demo')
else:
    bot, top = demo['bottom'], demo['top']
    size = top - bot
    bull = demo['direction'] == 'bull'
    wick_col = 'low' if bull else 'high'
    idx  = demo['idx']

    mid_bar   = htf_demo.iloc[idx - 1]
    mid_range = mid_bar['high'] - mid_bar['low']
    mid_body  = abs(mid_bar['close'] - mid_bar['open'])
    body_ratio = mid_body / mid_range if mid_range > 0 else 0.0
    is_disp   = is_displacement_candle(htf_demo, idx, _DISP_MULT, _BODY_RATIO)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), facecolor=BG)

    # ── Left: schematic ───────────────────────────────────────────────────
    ax = axes[0]
    ax.set_facecolor(BG2); ax.set_xlim(0, 10); ax.set_ylim(-0.15, 1.20); ax.axis('off')
    color = GREEN if bull else RED
    ax.add_patch(mpatches.Rectangle((1, 0), 5, 1, fc=color, ec=color, alpha=0.18, zorder=1))
    ax.hlines([0, 1], 1, 6, colors=color, lw=1.4, ls='--', zorder=2)
    ax.text(6.2, 0, f'bottom = {bot:.2f}', color=color, fontsize=7.5, va='center')
    ax.text(6.2, 1, f'top    = {top:.2f}', color=color, fontsize=7.5, va='center')
    for d, c, lbl in [(0.0, FG, 'depth=0%   edge'), (0.2, GOLD, 'depth=20%  filter threshold'),
                      (0.5, BLUE, 'depth=50%'), (1.0, PURPLE, 'depth=100% wick fully through')]:
        y_norm = (1 - d) if bull else d
        ax.hlines(y_norm, 1, 6, colors=c, lw=1.5, zorder=3)
        ax.text(1.15, y_norm + 0.04, lbl, color=c, fontsize=7.5, va='bottom')
    wick_lbl = 'bar_low (lower wick)  ↓' if bull else 'bar_high (upper wick)  ↑'
    ax.text(3.5, -0.10, wick_lbl, color=GOLD, fontsize=7.5, ha='center', va='top')

    disp_tag = (
        f'displacement ✓  range={mid_range:.2f} (×{_DISP_MULT})  '
        f'body={body_ratio:.0%} (≥{_BODY_RATIO:.0%})'
        if is_disp else
        f'displacement ✗  range={mid_range:.2f}  body={body_ratio:.0%}'
    )
    ax.set_title(f'FVG Entry Depth — {demo["direction"]} FVG\n{disp_tag}',
                 color=CYAN if is_disp else RED, fontsize=8.5, fontweight='bold', pad=6)

    # ── Right: 3-bar pattern + deepest wick ──────────────────────────────
    ax2 = axes[1]
    ws  = max(0, idx - 20)
    we  = min(len(htf_demo), idx + 30)
    win = htf_demo.iloc[ws:we].reset_index(drop=True)
    draw_candles(ax2, win, x0=ws)
    style_ax(ax2, f'Actual FVG (bar {idx}) — displacement + deepest wick')
    time_ticks(ax2, win, step=8, x0=ws)
    ax2.add_patch(mpatches.Rectangle(
        (idx - 2, bot), len(htf_demo) - (idx - 2), size,
        fc=color, ec=color, alpha=0.18, lw=0.5))
    ax2.hlines([bot, top], idx - 2, ws + len(win), colors=color, lw=0.8, ls='--')

    # Highlight the 3-bar FVG pattern (A, B=displacement, C)
    for bar_abs, lbl_txt, ec_col in [(idx-2,'A',GRID),(idx-1,'B',CYAN),(idx,'C',GRID)]:
        bar_j = bar_abs - ws
        if 0 <= bar_j < len(win):
            r  = win.iloc[bar_j]
            lo = min(r['open'], r['close'])
            hi = max(r['open'], r['close'])
            ax2.add_patch(mpatches.Rectangle(
                (bar_abs - 0.45, lo), 0.90, max(hi - lo, 1e-6),
                fill=False, ec=ec_col, lw=1.8, zorder=7))
            ax2.text(bar_abs, r['high'] * 1.002, lbl_txt,
                     color=ec_col, fontsize=6.5, ha='center', va='bottom', fontweight='bold')

    # Deepest qualifying wick after FVG formation
    fvg_j = idx - ws
    best_j, best_depth, best_wick = -1, -1.0, None
    for j in range(fvg_j + 1, len(win)):
        row = win.iloc[j]
        wick_price = row['low'] if bull else row['high']
        overlaps = (bull     and wick_price <= top and row['high'] >= bot) or \
                   (not bull and wick_price >= bot and row['low']  <= top)
        if overlaps:
            d = fvg_entry_depth(demo, wick_price)
            if _MIN_DEPTH <= d <= _MAX_DEPTH and d > best_depth:
                best_j, best_depth, best_wick = j, d, wick_price

    if best_j >= 0:
        xb = ws + best_j
        ax2.plot(xb, best_wick, 'D', color=GOLD, ms=6, zorder=8)
        va  = 'top' if bull else 'bottom'
        off = best_wick * (0.997 if bull else 1.003)
        ax2.text(xb, off, f'depth={best_depth:.0%}',
                 color=GOLD, fontsize=6.5, ha='center', va=va, fontweight='bold')

    ax2.set_ylabel('Price', fontsize=7)
    legend_items = [
        mpatches.Patch(fc='none', ec=CYAN, lw=1.8, label='B: displacement candle'),
        mpatches.Patch(fc='none', ec=GRID, lw=1.8, label='A / C: surrounding bars'),
        matplotlib.lines.Line2D([0], [0], marker='D', color='w', markerfacecolor=GOLD,
                                ms=6, label=f'deepest wick ({wick_col}) depth {_MIN_DEPTH:.0%}–{_MAX_DEPTH:.0%}'),
    ]
    leg = ax2.legend(handles=legend_items, fontsize=6, framealpha=0.3, labelcolor=FG, loc='upper left')
    leg.get_frame().set_facecolor(BG2)

    plt.suptitle('Step 5: FVG Entry Depth (wick-based)', color=FG, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'FVG bar {idx}: bottom={bot:.3f}  top={top:.3f}  size={size:.4f}  dir={demo["direction"]}')
    print(f'Middle candle: range={mid_range:.3f}  body={mid_body:.3f}  body_ratio={body_ratio:.2f}  displacement={is_disp}')
    if best_j >= 0:
        print(f'Deepest wick : bar {ws+best_j}  {wick_col}={best_wick:.3f}  depth={best_depth:.2f}')
    else:
        print('No qualifying wick entry found in window')
    print('\nDepth reference table:')
    for d in [0.0, 0.2, 0.5, 1.0]:
        p = (top - d * size) if bull else (bot + d * size)
        print(f'  depth={d:.1f}  {wick_col}={p:.3f}  -> fvg_entry_depth={fvg_entry_depth(demo, p):.2f}')


## Step 6 — LTF 入场确认 (`check_ltf_confirmation`)

当影线触及 HTF FVG 后，在**低时间框架**等待反转信号：

1. **LTF CHoCH** → 短期结构反转（必须在影线进入 FVG **之后**出现）
2. **LTF BOS** → CHoCH 之后的顺势突破，确认方向

两个信号都必须在影线首次触及 FVG 之后出现，不接受提前的信号。  
这保证了逻辑一致性：回调必须真正触及 FVG，才能开始等待反转确认。

金色高亮区 = 完整的 CHoCH → BOS 确认序列。

In [ ]:
# ── Anchor on the first real trade ───────────────────────────────────────────
if not result.trades:
    print('No trades — cannot show FVG anchor. Run with looser params first.')
else:
    trade = result.trades[0]
    t_dir   = trade.direction
    t_entry = trade.entry_time

    htf_times = htf_raw["time_key"].values.astype(str)
    htf_pos   = int(np.searchsorted(htf_times, t_entry, side="right")) - 1

    ltf_times = ltf_raw["time_key"].values.astype(str)
    entry_ltf = int(np.searchsorted(ltf_times, t_entry, side="right")) - 1
    ws = max(0, entry_ltf - 60)
    we = min(len(ltf_raw), entry_ltf + 20)
    ltf = ltf_raw.iloc[ws:we].reset_index(drop=True)
    ltf_sigs = detect_bos_choch(ltf, lookback=1)

    ltf_high_max = ltf["high"].max()
    ltf_low_min  = ltf["low"].min()

    # Search anchor FVG — priority: active; fallback: filled
    anchor_fvg    = None
    anchor_filled = False
    htf_view      = None
    for htf_window_size in [50, 100, 200]:
        htf_start = max(0, htf_pos + 1 - htf_window_size)
        htf_view  = htf_raw.iloc[htf_start : htf_pos + 1].reset_index(drop=True)
        htf_fvgs  = detect_fvg(htf_view, min_gap_pct=0.002)
        in_range  = [f for f in htf_fvgs
                     if f["direction"] == t_dir
                     and f["bottom"] <= ltf_high_max
                     and f["top"]    >= ltf_low_min]
        active = [f for f in in_range if not f["filled"]]
        if active:
            anchor_fvg = min(active, key=lambda f: abs(trade.entry_price - (f["top"]+f["bottom"])/2))
            break
        if in_range:
            anchor_fvg    = min(in_range, key=lambda f: abs(trade.entry_price - (f["top"]+f["bottom"])/2))
            anchor_filled = True
            break

    # First LTF wick touch of the FVG zone
    fvg_touch_j = -1
    if anchor_fvg is not None and htf_view is not None:
        fvg_htf_time = htf_view.iloc[anchor_fvg["idx"]]["time_key"]
        for j, row in ltf.iterrows():
            if row["time_key"] <= fvg_htf_time:
                continue
            wick = row["low"] if t_dir == "bull" else row["high"]
            in_zone = (
                (t_dir == "bull" and wick <= anchor_fvg["top"]   and row["high"] >= anchor_fvg["bottom"]) or
                (t_dir == "bear" and wick >= anchor_fvg["bottom"] and row["low"]  <= anchor_fvg["top"])
            )
            if in_zone:
                fvg_touch_j = j
                break

    # CHoCH + BOS after FVG touch
    choch_x = bos_x = -1
    for sig in ltf_sigs:
        if sig["type"] == "CHoCH" and sig["direction"] == t_dir:
            if fvg_touch_j < 0 or sig["idx"] > fvg_touch_j:
                choch_x = sig["idx"]
        if choch_x >= 0 and sig["type"] == "BOS" and sig["direction"] == t_dir and sig["idx"] > choch_x:
            bos_x = sig["idx"]
            break

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
    draw_candles(ax, ltf, x0=ws)
    style_ax(ax, f'Step 6: LTF 15m CHoCH + BOS Confirmation  (HTF trend={t_dir})')
    time_ticks(ax, ltf, step=10, x0=ws, fmt='%m/%d %H:%M')

    if anchor_fvg is not None:
        fvg_color = GREEN if t_dir == "bull" else RED
        ax.axhspan(anchor_fvg["bottom"], anchor_fvg["top"],
                   facecolor=fvg_color, alpha=0.10 if anchor_filled else 0.18, zorder=1)
        ax.hlines([anchor_fvg["bottom"], anchor_fvg["top"]],
                  ws, ws + len(ltf), colors=fvg_color, lw=1.2, ls='--', alpha=0.8)
        mid_fvg = (anchor_fvg["top"] + anchor_fvg["bottom"]) / 2
        tag = f'HTF FVG  {anchor_fvg["bottom"]:.2f}–{anchor_fvg["top"]:.2f}'
        if anchor_filled:
            tag += '  (filled)'
        ax.text(ws + 0.5, mid_fvg, tag, color=fvg_color, fontsize=6.5,
                va='center', fontweight='bold',
                bbox=dict(fc=BG2, ec='none', alpha=0.7, pad=1))
    else:
        ax.text(0.5, 0.7, 'HTF FVG not found in price range',
                transform=ax.transAxes, color=RED, fontsize=8, ha='center')

    if fvg_touch_j >= 0:
        row    = ltf.iloc[fvg_touch_j]
        wick_p = row["low"] if t_dir == "bull" else row["high"]
        ax.plot(ws + fvg_touch_j, wick_p, '*', color=GOLD, ms=11, zorder=9)
        va  = 'top' if t_dir == "bull" else 'bottom'
        ax.text(ws + fvg_touch_j, wick_p * (0.997 if t_dir == "bull" else 1.003),
                'FVG touch', color=GOLD, fontsize=6, ha='center', va=va, fontweight='bold')

    # BOS / CHoCH lines — both labelled at line midpoint with gap filter
    _MIN_LABEL_GAP = 8
    last_label = {'bull': -999, 'bear': -999}
    for sig in ltf_sigs:
        bull  = sig['direction'] == 'bull'
        color = GREEN if bull else RED
        ls    = '-' if sig['type'] == 'BOS' else '--'
        lw    = 0.8 if sig['type'] == 'BOS' else 1.2
        ax.hlines(sig['price'], ws + sig['from_idx'], ws + sig['idx'],
                  colors=color, linestyles=ls, linewidths=lw, alpha=0.65, zorder=4)
        dir_ = sig['direction']
        if sig['idx'] - last_label[dir_] >= _MIN_LABEL_GAP:
            mid_x = ws + (sig['from_idx'] + sig['idx']) / 2
            off   = sig['price'] * 0.0015 * (1 if bull else -1)
            ax.text(mid_x, sig['price'] + off, sig['type'],
                    color=color, fontsize=5.5, ha='center',
                    va='bottom' if bull else 'top', fontweight='bold')
            last_label[dir_] = sig['idx']

    if bos_x > 0:
        ax.axvspan(ws + choch_x - 1, ws + bos_x + 1, facecolor=GOLD, alpha=0.10)
        ax.axvline(ws + choch_x, color=GOLD, lw=1.0, ls=':', alpha=0.7)
        ax.axvline(ws + bos_x,   color=GOLD, lw=1.0, ls=':', alpha=0.7)
        ax.text(ws + (choch_x + bos_x) / 2, ax.get_ylim()[1] * 0.999,
                f'CHoCH@{ws+choch_x} → BOS@{ws+bos_x}',
                color=GOLD, fontsize=7.5, ha='center', va='top', fontweight='bold')

    ax.axvline(entry_ltf, color=BLUE, lw=1.3, ls=':', alpha=0.9)
    ax.text(entry_ltf + 0.4, ax.get_ylim()[0],
            f'ENTRY\n{trade.entry_price:.2f}',
            color=BLUE, fontsize=6.5, va='bottom', fontweight='bold')

    items = [
        mpatches.Patch(fc=GREEN if t_dir=='bull' else RED, alpha=0.3,
                       label=f'HTF FVG anchor ({t_dir})'),
        matplotlib.lines.Line2D([0],[0], marker='*', color='w',
                                markerfacecolor=GOLD, ms=9, label='First FVG touch'),
        mpatches.Patch(fc=GREEN, alpha=0.5, label='Bull signal (BOS solid / CHoCH dashed)'),
        mpatches.Patch(fc=RED,   alpha=0.5, label='Bear signal (BOS solid / CHoCH dashed)'),
        mpatches.Patch(fc=GOLD,  alpha=0.2, label='CHoCH → BOS window'),
    ]
    leg = ax.legend(handles=items, fontsize=6.5, framealpha=0.3, labelcolor=FG, loc='upper left')
    leg.get_frame().set_facecolor(BG2)
    ax.set_xlabel('15m bar', fontsize=7)
    ax.set_ylabel('Price', fontsize=7)
    plt.tight_layout()
    plt.show()

    print(f'Trade #1 : {t_dir}  entry={t_entry}  price={trade.entry_price:.3f}')
    if anchor_fvg:
        print(f'HTF FVG  : {anchor_fvg["bottom"]:.3f}–{anchor_fvg["top"]:.3f}'
              f'  {"(filled)" if anchor_filled else "(active)"}')
    print(f'FVG touch: LTF bar {ws+fvg_touch_j if fvg_touch_j>=0 else "not found"}')
    if bos_x > 0:
        print(f'Confirmed: CHoCH@{ws+choch_x}  BOS@{ws+bos_x}')


## Step 7 — 完整交易设置

| 要素 | 牛市 | 熊市 |
|------|------|------|
| **入场** | LTF BOS 确认 K 线收盘价 | 同左 |
| **止损 SL** | 最近 HTF 摆动低点 × (1 − buffer) | 最近 HTF 摆动高点 × (1 + buffer) |
| **止盈 TP** | 最近 HTF 摆动高点（取最低的一个） | 最近 HTF 摆动低点（取最高的一个） |
| **过滤** | SL% ≤ max_sl_pct，RR ≥ min_rr | 同左 |

In [ ]:
if result.trades:
    trade = result.trades[0]
    entry, sl, tp = trade.entry_price, trade.sl, trade.tp
    direction     = trade.direction
    pos = int((htf_raw['time_key'] <= trade.entry_time).sum()) - 1
    ws  = max(0, pos - 35)
    we  = min(len(htf_raw), pos + 25)
    _htf = htf_raw.iloc[ws:we].reset_index(drop=True)
    entry_bar = pos - ws
    print(f'Trade: {direction}  entry_time={trade.entry_time}')
    print(f'  entry={entry:.3f}  SL={sl:.3f}  TP={tp:.3f}')
    print(f'  RR={trade.planned_rr:.2f}  Result={trade.result}  R={trade.r_multiple:+.2f}')
else:
    _htf = htf.copy()
    mid  = len(_htf) // 2
    entry = _htf.iloc[mid]['close']
    sl    = _htf.iloc[mid]['close'] * 0.97
    tp    = _htf.iloc[mid]['close'] * 1.06
    direction = 'bull'; entry_bar = mid
    print('No real trade found -- using synthetic example')

sl_dist = abs(entry - sl)
rr      = abs(tp - entry) / sl_dist if sl_dist > 0 else 0

fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
draw_candles(ax, _htf)
style_ax(ax, 'Step 7: Full Trade Setup -- Entry / SL / TP')
time_ticks(ax, _htf, step=8)

ax.axhline(entry, color=BLUE,  lw=1.6, label=f'Entry  {entry:.3f}')
ax.axhline(sl,    color=RED,   lw=1.4, ls='--', label=f'SL  {sl:.3f}  (1R = {sl_dist:.3f})')
ax.axhline(tp,    color=GREEN, lw=1.4, ls='--', label=f'TP  {tp:.3f}  (RR = {rr:.2f})')
ax.axvline(entry_bar, color=BLUE, lw=1.2, ls=':')
ax.text(entry_bar + 0.5, entry * 1.001, 'ENTRY', color=BLUE, fontsize=8, fontweight='bold')

ax.fill_between(range(len(_htf)), sl,    entry, color=RED,   alpha=0.07)
ax.fill_between(range(len(_htf)), entry, tp,    color=GREEN, alpha=0.07)

xr = len(_htf) - 2
ax.annotate('', xy=(xr, sl), xytext=(xr, entry),
            arrowprops=dict(arrowstyle='<->', color=RED, lw=1.2))
ax.text(xr + 0.3, (sl + entry) / 2, '1R', color=RED, fontsize=7, va='center')
ax.annotate('', xy=(xr, tp), xytext=(xr, entry),
            arrowprops=dict(arrowstyle='<->', color=GREEN, lw=1.2))
ax.text(xr + 0.3, (tp + entry) / 2, f'{rr:.1f}R', color=GREEN, fontsize=7, va='center')

leg = ax.legend(fontsize=7.5, framealpha=0.3, labelcolor=FG, loc='upper left')
leg.get_frame().set_facecolor(BG2)
ax.set_ylabel('Price', fontsize=7)
plt.tight_layout()
plt.show()


## 综合演示 — 多参数对比 + 权益曲线

In [ ]:
PSETS = [
    dict(fvg_min_width_pct=0.001, fvg_entry_depth_pct=0.10, min_rr=1.5),
    dict(fvg_min_width_pct=0.002, fvg_entry_depth_pct=0.20, min_rr=1.5),
    dict(fvg_min_width_pct=0.002, fvg_entry_depth_pct=0.50, min_rr=2.0),
    dict(fvg_min_width_pct=0.005, fvg_entry_depth_pct=0.50, min_rr=2.0),
]
COLORS4 = [GREEN, BLUE, GOLD, PURPLE]

fig, axes = plt.subplots(1, 2, figsize=(16, 6.0), facecolor=BG)
fig.subplots_adjust(bottom=0.18)
ax_eq  = axes[0]
ax_tbl = axes[1]
style_ax(ax_eq, 'Equity Curves (Cumulative R)')
ax_eq.axhline(0, color=GRID, lw=0.8)

rows = []
for ps, color in zip(PSETS, COLORS4):
    p = BacktestParams(trend_tf='60m', entry_tf='15m',
                       swing_lookback=2, bos_count=1,
                       sl_buffer_pct=0.001, max_sl_pct=0.010, **ps)
    r = run_backtest(htf_raw, ltf_raw, p)
    lbl = (f"w={ps['fvg_min_width_pct']:.3f}  "
           f"dp={ps['fvg_entry_depth_pct']:.2f}  "
           f"rr{ps['min_rr']:.1f}  "
           f"T={r.n_trades}  PF={r.profit_factor:.2f}")
    if r.trades:
        eq = np.concatenate([[0], np.cumsum([t.r_multiple for t in r.trades])])
        ax_eq.plot(eq, color=color, lw=1.6, label=lbl)
    else:
        ax_eq.plot([0], color=color, lw=1, ls=':', label=lbl + '  (no trades)')
    rows.append([ps['fvg_min_width_pct'], ps['fvg_entry_depth_pct'], ps['min_rr'],
                 r.n_trades, f'{r.win_rate:.0%}',
                 f'{r.total_r:+.2f}', f'{r.profit_factor:.2f}', f'{r.max_drawdown_r:.2f}'])

leg = ax_eq.legend(fontsize=6.5, framealpha=0.3, labelcolor=FG, loc='upper left')
leg.get_frame().set_facecolor(BG2)
ax_eq.set_xlabel('Trade #', fontsize=7)
ax_eq.set_ylabel('Cumulative R', fontsize=7)

# ── Table with input-param vs result column colour coding ────────────────────
ax_tbl.set_facecolor(BG2); ax_tbl.axis('off')
_IN  = {0, 1, 2}        # input parameters
_OUT = {3, 4, 5, 6, 7}  # backtest results
_HDR_IN   = '#1a3050'
_HDR_OUT  = '#1a3028'
_CELL_IN  = '#111a28'
_CELL_OUT = BG2

cols = ['min_w', 'depth', 'min_rr', 'trades', 'WR', 'Total R', 'PF', 'MaxDD']
tbl  = ax_tbl.table(cellText=rows, colLabels=cols, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.1, 1.9)
for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(GRID)
    if row == 0:
        cell.set_facecolor(_HDR_IN if col in _IN else _HDR_OUT)
        cell.set_text_props(color=CYAN if col in _IN else GREEN, fontweight='bold')
    else:
        cell.set_facecolor(_CELL_IN if col in _IN else _CELL_OUT)
        cell.set_text_props(color=CYAN if col in _IN else FG)
ax_tbl.set_title('Parameter Comparison', color=FG, fontsize=9, fontweight='bold', pad=10)

# ── Abbreviation legend at figure bottom ─────────────────────────────────────
_abbr = (
    "Inputs (blue):   min_w = min FVG width as fraction of price    "
    "depth = min wick penetration into FVG    "
    "min_rr = minimum risk/reward ratio\n"
    "Results (green): trades = number of triggered trades    "
    "WR = win rate    "
    "Total R = cumulative P&L in R units    "
    "PF = profit factor (gross win / gross loss)    "
    "MaxDD = max drawdown in R"
)
fig.text(0.5, 0.02, _abbr, color=FG, fontsize=7, ha='center', va='bottom',
         bbox=dict(fc=BG, ec=GRID, alpha=0.85, pad=5, boxstyle='round,pad=0.5'))

plt.suptitle('Combined: 60m/15m Parameter Sensitivity', color=FG, fontsize=11, fontweight='bold')
plt.tight_layout(rect=[0, 0.14, 1, 1])
plt.show()
